# Sesiones 15 y 16 · Dispersión

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

## 1 · Los datos

Los hogares de la ENIGH 2024, que ya conoces: cada renglón es un hogar, y
`ingreso` y `gasto` están en pesos por trimestre. También las acciones de Apple
(AAPL) y Tesla (TSLA), con su precio diario en dólares.

In [ ]:
library(tidyverse)
source("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/R/colores.R")   # colores del curso

hogares <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/enigh2024_hogares.csv.gz", show_col_types = FALSE)
mercado <- read_csv("https://raw.githubusercontent.com/cjjmdata/analisis_datos_i/main/datos/mercado_diario.csv", show_col_types = FALSE)

## 2 · El mismo centro, otra dispersión

Durango y Michoacán tienen casi el mismo ingreso medio. Antes de correr la
siguiente celda: ¿en cuál de los dos estados crees que los ingresos están más
separados entre sí?

In [ ]:
dos <- filter(hogares, entidad %in% c("Durango", "Michoacán"))

dos |>
  group_by(entidad) |>
  summarize(hogares = n(), media = mean(ingreso), mediana = median(ingreso))

In [ ]:
ggplot(dos, aes(x = ingreso, y = entidad)) +
  geom_boxplot(fill = TEAL_100, color = TEAL_700, outlier.color = col_destaca, outlier.alpha = 0.5) +
  coord_cartesian(xlim = c(0, 300000)) +
  scale_x_continuous(labels = scales::comma) +
  labs(x = "Ingreso trimestral del hogar, en pesos", y = NULL)

## 3 · Rango y rango intercuartil

Las dos medidas se calculan por posición: el rango con el mínimo y el máximo, el
rango intercuartil con los cuartiles 1 y 3.

In [ ]:
diff(range(hogares$ingreso))     # máximo menos mínimo

IQR(hogares$ingreso)             # tercer cuartil menos primer cuartil

Si quitas el hogar de mayor ingreso, ¿cuál de las dos cambia más? Anota tu
respuesta y corre la celda.

In [ ]:
sin_mayor <- filter(hogares, ingreso < max(ingreso))

diff(range(sin_mayor$ingreso))
IQR(sin_mayor$ingreso)

## 4 · Varianza y desviación estándar

Las edades del jefe del hogar en los cinco primeros hogares de la base. Con tu
calculadora: la media, la distancia de cada edad a la media y la suma de esas
distancias.

In [ ]:
edades <- hogares$edad_jefe[1:5]
edades

desv <- edades - mean(edades)
desv
sum(desv)

### Primero a mano

La varianza eleva cada desviación al cuadrado y divide la suma entre $n - 1$:

$$s^2 = \frac{\sum (x_i - \bar{x})^2}{n - 1}$$

¿Coinciden las tres cifras de la celda siguiente?

In [ ]:
sum(desv^2) / (length(edades) - 1)    # la fórmula, traducida literalmente

var(edades)                           # la función de R

sum(desv^2) / length(edades)          # dividiendo entre n

La desviación estándar es la raíz de la varianza y regresa a la unidad original:
de años al cuadrado, a años.

In [ ]:
sqrt(var(edades))
sd(edades)

Cambia una de las edades por 90 y vuelve a calcular. ¿Cuánto cambia la
desviación estándar?

In [ ]:
otras <- edades
otras[3] <- 90

sd(edades)
sd(otras)

### La desviación estándar del ingreso

¿Cuánto crees que cambia la desviación estándar de los hogares del país si quitas
el hogar de mayor ingreso?

In [ ]:
sd(hogares$ingreso)
sd(sin_mayor$ingreso)

IQR(hogares$ingreso)
IQR(sin_mayor$ingreso)

Durango y Michoacán con las dos medidas. ¿Cuál es más disperso con cada una?

In [ ]:
dos |>
  group_by(entidad) |>
  summarize(media = mean(ingreso), de = sd(ingreso), ric = IQR(ingreso), maximo = max(ingreso))

## 5 · Coeficiente de variación

La desviación estándar como porcentaje de la media:

$$CV = \frac{s}{\bar{x}} \times 100$$

¿Qué varía más entre los hogares, el ingreso o el número de integrantes?

In [ ]:
sd(hogares$ingreso) / mean(hogares$ingreso) * 100

sd(hogares$integrantes) / mean(hogares$integrantes) * 100

### Precios de dos acciones en 2025

La desviación estándar del precio de TSLA es mayor que la de AAPL. ¿Su precio fue
más disperso también en relación con su precio medio?

In [ ]:
mercado |>
  filter(year(fecha) == 2025) |>
  group_by(clave) |>
  summarize(media = mean(cierre), de = sd(cierre), cv = de / media * 100)

Ahora con los rendimientos diarios, en porcentaje. Según el coeficiente de
variación, ¿cuál acción fue más arriesgada? ¿Coincide con la desviación
estándar?

In [ ]:
mercado |>
  arrange(clave, fecha) |>
  group_by(clave) |>
  mutate(rend = (ajustado / lag(ajustado) - 1) * 100) |>
  filter(year(fecha) == 2025) |>
  summarize(media = mean(rend), de = sd(rend), cv = de / media * 100)

La media de un rendimiento diario está cerca de cero, y dividir entre un número
cercano a cero produce cocientes enormes. El coeficiente de variación solo tiene
sentido con variables positivas cuyo cero significa ausencia.

## 6 · Dispersión por grupos

`group_by()` separa la base en grupos y `summarize()` calcula un renglón por
grupo.

In [ ]:
hogares |>
  group_by(tam_loc) |>
  summarize(
    hogares = n(),
    media   = mean(ingreso),
    de      = sd(ingreso),
    cv      = de / media * 100,
    ric     = IQR(ingreso)
  )

Los ingresos de los hogares de Oaxaca, ¿están más o menos dispersos que los del
resto del país?

In [ ]:
hogares |>
  mutate(region = if_else(entidad == "Oaxaca", "Oaxaca", "Resto del país")) |>
  group_by(region) |>
  summarize(media = mean(ingreso), de = sd(ingreso),
            cv = de / media * 100, ric = IQR(ingreso))

In [ ]:
cv_entidad <- hogares |>
  group_by(entidad) |>
  summarize(cv = sd(ingreso) / mean(ingreso) * 100) |>
  arrange(desc(cv))

cv_entidad

## 7 · Tu serie

Sobre la variable numérica de tu serie del portafolio: el rango, el rango
intercuartil, la desviación estándar y el coeficiente de variación, con sus
unidades. Si tu serie es un índice o un precio que sigue una tendencia, calcula
la dispersión de sus variaciones de un periodo al siguiente.

In [ ]:
# Lectura de tu serie
